# LSOT / MW2 3DGS experiment on a CUDA machine

This launcher creates a Python 3.11 environment even if the notebook host uses Python 3.13. Select a CUDA GPU runtime. Supply your own Train/Truck datasets and trusted checkpoints. Real-scene GPU validation is required; the numerical tests do not replace it.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json
REPO = Path('/content/LSOT_GMM')
REVISION = 'main'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/tmp0810/LSOT_GMM.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REVISION], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', REVISION], check=True)
subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', REVISION], check=True)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
# A separate interpreter avoids the host notebook's Python/CUDA dependency conflicts.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
VENV = Path('/content/lsot3dgs_env')
if not (VENV / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--seed', '--python', '3.11', str(VENV)], check=True)
PYTHON = str(VENV / 'bin/python')
ENV = {**os.environ, 'PATH': str(VENV / 'bin') + ':' + os.environ['PATH'],
       'TORCH_CUDA_ARCH_LIST': '8.0;9.0', 'MAX_JOBS': '4'}
# On a managed server install Eigen via its package manager instead.
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'build-essential', 'libeigen3-dev'], check=True)
subprocess.run(['bash', str(REPO / '3DGS/setup_env.sh'), '--install-torch'], cwd=REPO, env=ENV, check=True)


## Set the real input paths

Put each COLMAP scene under `DATA/train` or `DATA/truck`, containing `images/` and `sparse/0/`. Put checkpoints under `DATA/checkpoints`. Copy from Drive to local disk for faster IO if needed. These assets are not part of the repository.


In [ ]:
DATA = Path('/content/data/3dgs')  # EDIT THIS
CONFIG = REPO / '3DGS/local_colab.json'
config = json.loads((REPO / '3DGS/configs/hmw_protocol.json').read_text())
for name, iteration in [('train', 30000), ('truck', 15000)]:
    config['scenes'][name]['dataset'] = str(DATA / name)
    config['scenes'][name]['checkpoint'] = str(DATA / 'checkpoints' / f'{name}_{iteration}.pth')
config['output_dir'] = str(REPO / 'results/3dgs_full')
CONFIG.write_text(json.dumps(config, indent=2))
def run(script, *args):
    subprocess.run([PYTHON, str(REPO / '3DGS' / script), *map(str, args)], cwd=REPO, env=ENV, check=True)
run('prepare_data.py', '--config', CONFIG)
run('preflight.py', '--config', CONFIG, '--renderer', '--scene', 'train')


In [ ]:
# Inspect the first compressed images before running the complete matrix.
run('run.py', '--config', CONFIG, '--scenes', 'train', '--retentions', '0.1',
    '--methods', 'mw2', 'min-B1D', 'ghap', 'hmw-local', '--no-finetune',
    '--output-dir', REPO / 'results/3dgs_pilot_pre')
run('plot.py', '--results-dir', REPO / 'results/3dgs_pilot_pre', '--scene', 'train',
    '--retention', '0.1', '--phase', 'pre', '--views', '0')


In [ ]:
# Full configured scene experiment, with pre/post metrics.
run('run.py', '--config', CONFIG)
run('plot.py', '--results-dir', config['output_dir'])


In [ ]:
# Optional: all eight LSOT variants plus MW2.
# run('run.py', '--config', CONFIG, '--methods', 'mw2',
#     'min-Mix', 'min-SMix', 'min-B', 'min-B1D',
#     'avg-Mix', 'avg-SMix', 'avg-B', 'avg-B1D',
#     '--output-dir', REPO / 'results/3dgs_all9')


In [ ]:
# Independent solver-scaling experiment on real Gaussian subsets.
SCALING = REPO / '3DGS/local_scaling.json'
scaling = json.loads((REPO / '3DGS/configs/scaling.json').read_text())
for name in scaling['scenes']:
    scaling['scenes'][name]['checkpoint'] = config['scenes'][name]['checkpoint']
scaling['output_dir'] = str(REPO / 'results/3dgs_scaling')
SCALING.write_text(json.dumps(scaling, indent=2))
run('benchmark.py', '--config', SCALING)
run('plot.py', '--results-dir', scaling['output_dir'])


Results on `/content` are temporary. Copy the result directories to persistent storage before ending the runtime. Reusing a result requires identical configuration, input hashes, code and runtime fingerprint; changed settings should use a new output directory.
